
# 📘 階層型インデックス 実務活用ノート（解答なし）
**対象**：Pandas 第8章「データラングリング：連結、結合、変形」  
**テーマ**：MultiIndex（階層型インデックス）の実務活用  
**作成日**：2025-10-13（JST）

> すべての問題は **コメントアウト形式** の指示に従ってコードセルを編集・実行してください。  
> ライブラリのインポートやサンプルデータは各セクションに用意してあります。必要に応じて追記/変更OK。

---


In [1]:

# === Setup ===
# このノート全体で使う基本ライブラリを読み込みます。
import pandas as pd
import numpy as np

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)

# 乱数の再現性
rng = np.random.default_rng(42)



## 1. 基本：階層型インデックスの作成とアクセス
- 目的：`set_index` で MultiIndex を作成し、`loc` でのアクセス、スライス、部分抽出を体験します。


In [5]:

# === Q1: 階層型インデックスの作成 ===
# 次のデータから、「store」「category」を行インデックスに設定して MultiIndex DataFrame を作成してください。
# その後、index の names を ["store", "category"] に設定してください。

df = pd.DataFrame({
    "store": ["Tokyo", "Tokyo", "Tokyo", "Osaka", "Osaka", "Nagoya", "Nagoya"],
    "category": ["A", "B", "C", "A", "B", "A", "C"],
    "sales": [100, 120, 90, 130, 95, 110, 105],
    "qty":   [10,   12,  8,  13,  9,  11,  10]
})

# ここに処理を書いて、結果を表示してください。
df_MultiIndex = df.set_index(["store","category"])

print(df)
print(df_MultiIndex)

df_MultiIndex.index.names = ["store","category"]

print(df_MultiIndex)


    store category  sales  qty
0   Tokyo        A    100   10
1   Tokyo        B    120   12
2   Tokyo        C     90    8
3   Osaka        A    130   13
4   Osaka        B     95    9
5  Nagoya        A    110   11
6  Nagoya        C    105   10
                 sales  qty
store  category            
Tokyo  A           100   10
       B           120   12
       C            90    8
Osaka  A           130   13
       B            95    9
Nagoya A           110   11
       C           105   10
                 sales  qty
store  category            
Tokyo  A           100   10
       B           120   12
       C            90    8
Osaka  A           130   13
       B            95    9
Nagoya A           110   11
       C           105   10


In [15]:

# === Q2: 単一キーと複数キーでのアクセス ===
# Q1で作成した MultiIndex DataFrame から、
# 1) store="Tokyo" の全行
# 2) store="Nagoya", category="C" の行
# 3) store in ["Tokyo", "Osaka"] かつ category in ["A","B"] の部分抽出
# をそれぞれ loc を使って取得・表示してください。

# ここに処理を書いてください。
print(df_MultiIndex.loc["Tokyo"])
print(df_MultiIndex.loc["Nagoya","C"])
print(df_MultiIndex.loc[(["Tokyo","Osaka"],["A","B"]),:])


          sales  qty
category            
A           100   10
B           120   12
C            90    8
sales    105
qty       10
Name: (Nagoya, C), dtype: int64
                sales  qty
store category            
Tokyo A           100   10
      B           120   12
Osaka A           130   13
      B            95    9


In [101]:

# === Q3: スライスと並び替え ===
# MultiIndex はソートされていないとスライスの挙動が直感的でない場合があります。
# 1) index を sort_index() で昇順ソート
# 2) スライス記法を用いて、"Osaka"〜"Tokyo" の範囲を取り出してください。
# 3) category レベルで "A"〜"B" の範囲を取り出してください。

# ここに処理を書いてください。
df_MultiIndex_sort = df_MultiIndex.sort_index(ascending=True)

print(df_MultiIndex_sort)
print(df_MultiIndex_sort.loc["Osaka":"Tokyo"])

idx = pd.IndexSlice
tmp = df_MultiIndex.sort_index(level="category")
print(df_MultiIndex.loc[idx[:,["A","B"]],:])


                 sales  qty
store  category            
Nagoya A           110   11
       C           105   10
Osaka  A           130   13
       B            95    9
Tokyo  A           100   10
       B           120   12
       C            90    8
                sales  qty
store category            
Osaka A           130   13
      B            95    9
Tokyo A           100   10
      B           120   12
      C            90    8
                 sales  qty
store  category            
Tokyo  A           100   10
Osaka  A           130   13
Nagoya A           110   11
Tokyo  B           120   12
Osaka  B            95    9



## 2. 集計結果（pivot_table）との連携
- 目的：`pivot_table` で作成した集計結果の MultiIndex を操作します。


In [46]:

# === Q4: pivot_tableの作成 ===
# 次のトランザクションデータから、店舗×カテゴリ×年 で売上合計を集計し、
# 行: ["store", "category"], 列: "year", 値: "amount" のピボットテーブルを作成してください。
# aggfunc は "sum" とします。

tx = pd.DataFrame({
    "store":    ["Tokyo","Tokyo","Tokyo","Osaka","Osaka","Nagoya","Nagoya","Nagoya"],
    "category": ["A","A","B","B","A","A","B","B"],
    "year":     [2023,2024,2024,2023,2024,2023,2023,2024],
    "amount":   [100,120, 80,  90, 130, 110, 95,  85]
})

# ここに処理を書いてください。
pivot_tx = tx.pivot_table(index=["store","category"],columns="year",values="amount",aggfunc="sum")
print(tx)
print(pivot_tx)


    store category  year  amount
0   Tokyo        A  2023     100
1   Tokyo        A  2024     120
2   Tokyo        B  2024      80
3   Osaka        B  2023      90
4   Osaka        A  2024     130
5  Nagoya        A  2023     110
6  Nagoya        B  2023      95
7  Nagoya        B  2024      85
year              2023   2024
store  category              
Nagoya A         110.0    NaN
       B          95.0   85.0
Osaka  A           NaN  130.0
       B          90.0    NaN
Tokyo  A         100.0  120.0
       B           NaN   80.0


In [ ]:

# === Q5: 列レベルの抽出と再構成 ===
# Q4のピボットテーブルから、特定の年(例: 2024) の列だけを取り出してください。
# さらに、列に残った年のレベル名を "year" に設定して、見出しをわかりやすくしてください。

# ここに処理を書いてください。
pivot_tx_2024 = pivot_tx[2024]
#pivot_tx_2024
print(pivot_tx_2024)


store   category
Nagoya  A             NaN
        B            85.0
Osaka   A           130.0
        B             NaN
Tokyo   A           120.0
        B            80.0
Name: 2024, dtype: float64



## 3. 変形：`stack()` / `unstack()`
- 目的：列⇔行のレベルを移し替え、整形や再集計を容易にします。


In [ ]:

# === Q6: unstackで列展開 ===
# Q4で作成したピボットテーブル（または同等のDataFrame）を用い、
# 1) category を列方向に展開（unstack）する例
# 2) category を再び行方向に戻す（stack）例
# を試してください。level 引数を明示して、どのレベルを動かすのかコメントで説明してください。

# ここに処理を書いてください。
print(pivot_tx)
pivot_tx_unsta = pivot_tx.unstack(level="category")#カテゴリレベルを動かす
pivot_tx_sta = pivot_tx_unsta.stack(level="category")#カテゴリレベルを動かす
print(pivot_tx_unsta)
print(pivot_tx_sta)


year              2023   2024
store  category              
Nagoya A         110.0    NaN
       B          95.0   85.0
Osaka  A           NaN  130.0
       B          90.0    NaN
Tokyo  A         100.0  120.0
       B           NaN   80.0
year       2023         2024      
category      A     B      A     B
store                             
Nagoya    110.0  95.0    NaN  85.0
Osaka       NaN  90.0  130.0   NaN
Tokyo     100.0   NaN  120.0  80.0
year              2023   2024
store  category              
Nagoya A         110.0    NaN
       B          95.0   85.0
Osaka  A           NaN  130.0
       B          90.0    NaN
Tokyo  A         100.0  120.0
       B           NaN   80.0


C:\Users\kawor\AppData\Local\Temp\ipykernel_18180\1755120491.py:10: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  pivot_tx_sta = pivot_tx_unsta.stack()#カテゴリレベルを動かす


In [ ]:

# === Q7: MultiIndex Seriesのstack/unstack ===
# 次の Series は (store, month) をインデックスにもつ売上時系列です。
# 1) month を列に展開してワイド化（unstack）
# 2) 再びロング化（stack）
# 3) 展開・戻しで値が元に一致するかを "equals" で検証
# を行ってください。

months = pd.date_range("2024-01-01", periods=6, freq="MS")
mi = pd.MultiIndex.from_product([["Tokyo","Osaka"], months], names=["store","month"])
sales_s = pd.Series(rng.integers(80, 160, size=len(mi)), index=mi, name="sales")

# ここに処理を書いてください。
print(sales_s)
uns_sales = sales_s.unstack(level="month")
print(uns_sales)
sta_sales = uns_sales.stack(level="month").sort_index()
print(sta_sales)

print(sales_s.equals(sta_sales))


store  month     
Tokyo  2024-01-01    159
       2024-02-01    148
       2024-03-01     82
       2024-04-01     98
       2024-05-01    145
       2024-06-01     84
Osaka  2024-01-01    148
       2024-02-01    102
       2024-03-01    153
       2024-04-01    103
       2024-05-01    114
       2024-06-01    132
Name: sales, dtype: int64
month  2024-01-01  2024-02-01  2024-03-01  2024-04-01  2024-05-01  2024-06-01
store                                                                        
Osaka         148         102         153         103         114         132
Tokyo         159         148          82          98         145          84
store  month     
Tokyo  2024-06-01     84
       2024-05-01    145
       2024-04-01     98
       2024-03-01     82
       2024-02-01    148
       2024-01-01    159
Osaka  2024-06-01    132
       2024-05-01    114
       2024-04-01    103
       2024-03-01    153
       2024-02-01    102
       2024-01-01    148
dtype: int64
False



## 4. 時系列データでの階層構造
- 目的：`store × month` のような時系列MultiIndexでの抽出・集計を体験します。


In [85]:

# === Q8: 時系列抽出とグループ集計 ===
# Q7の sales_s を用いて、
# 1) store="Tokyo" のみを抽出
# 2) 各storeの月平均を計算（groupby(level="store").mean() など）
# 3) 2024-03 以降のデータに限定して可視化や集計（表示のみでOK）
# を行ってください。

# ここに処理を書いてください。
sales_s_tokyo = sales_s["Tokyo"]

print(sales_s_tokyo)
sales_s_store_marn = sales_s.groupby("store").mean()

print(sales_s_store_marn)

idx = pd.IndexSlice
t0 = pd.Timestamp("2024-03-01")
tmp = sales_s.sort_index()
out = tmp.loc[idx[:, t0:]]   # storeは全体(:)、monthは t0以降
print(out)


month
2024-01-01    159
2024-02-01    148
2024-03-01     82
2024-04-01     98
2024-05-01    145
2024-06-01     84
Freq: MS, Name: sales, dtype: int64
store
Osaka    125.333333
Tokyo    119.333333
Name: sales, dtype: float64
store  month     
Osaka  2024-03-01    153
       2024-04-01    103
       2024-05-01    114
       2024-06-01    132
Tokyo  2024-03-01     82
       2024-04-01     98
       2024-05-01    145
       2024-06-01     84
Name: sales, dtype: int64



## 5. 複数キーの結合と比較
- 目的：`store × year` の複合キーで予算と実績を突合します。


In [84]:

# === Q9: 複合キーJOIN（MultiIndex） ===
# 下の2つの DataFrame をそれぞれ set_index(["store","year"]) して MultiIndex 化し、
# left.join(right, how="outer") で結合してください。
# 結果の index.names を明示し、NaNを0で埋めた版も表示してください。

left = pd.DataFrame({
    "store": ["Tokyo", "Tokyo", "Osaka", "Nagoya"],
    "year":  [2023,    2024,    2024,    2023],
    "budget":[100,     120,     130,     110]
})

right = pd.DataFrame({
    "store": ["Tokyo", "Osaka", "Osaka", "Nagoya"],
    "year":  [2024,    2023,    2024,    2024],
    "sales": [115,     90,      125,     85 ]
})

# ここに処理を書いてください。
left2 = left.set_index(["store","year"])
right2 = right.set_index(["store","year"])
join_lr = left2.join(right2,how="outer")

print(join_lr)

join_lr = join_lr.fillna(0)

print(join_lr)

             budget  sales
store  year               
Nagoya 2023   110.0    NaN
       2024     NaN   85.0
Osaka  2023     NaN   90.0
       2024   130.0  125.0
Tokyo  2023   100.0    NaN
       2024   120.0  115.0
             budget  sales
store  year               
Nagoya 2023   110.0    0.0
       2024     0.0   85.0
Osaka  2023     0.0   90.0
       2024   130.0  125.0
Tokyo  2023   100.0    0.0
       2024   120.0  115.0



## 6. インデックスのソート・names 操作
- 目的：`sort_index` と `index/columns.names` を使って、読みやすく・処理しやすく整えます。


In [102]:

# === Q10: sort_index と names ===
# Q9で作成した結合結果に対して、
# 1) index を昇順に sort_index()
# 2) index.names を ["store","year"] に明示
# 3) 列名や columns.names（必要なら）も設定
# を行ってください。

# ここに処理を書いてください。

print(join_lr)

join_lr = join_lr.sort_index(ascending=True)

print(join_lr)

join_lr.index.names = ["store","year"]

print(join_lr)


             budget  sales
store  year               
Tokyo  2024   120.0  115.0
       2023   100.0    0.0
Osaka  2024   130.0  125.0
       2023     0.0   90.0
Nagoya 2024     0.0   85.0
       2023   110.0    0.0
             budget  sales
store  year               
Nagoya 2023   110.0    0.0
       2024     0.0   85.0
Osaka  2023     0.0   90.0
       2024   130.0  125.0
Tokyo  2023   100.0    0.0
       2024   120.0  115.0
             budget  sales
store  year               
Nagoya 2023   110.0    0.0
       2024     0.0   85.0
Osaka  2023     0.0   90.0
       2024   130.0  125.0
Tokyo  2023   100.0    0.0
       2024   120.0  115.0



## 7. 総合演習：店舗×カテゴリ×年の分析データ整形
- 目的：ここまでの要素（MultiIndex作成、stack/unstack、pivot_table、結合、集計）を組み合わせます。


In [ ]:

# === Q11: 総合整形フロー ===
# 次のランダム生成データを用いて、以下の処理を順に実施してください。
# (A) データ生成：店舗・カテゴリ・年でランダム売上（amount）を持つDataFrameを作成
# (B) 行インデックスを ["store","category","year"] に設定（MultiIndex）
# (C) pivot_table で 行=["store","category"], 列="year", 値="amount" の表を作成（agg=sum）
# (D) "year" 列の一部を抽出（例: 最新年のみ）し、列名や columns.names をわかりやすく設定
# (E) category を列に展開（unstack）→ 一旦ワイド化 → stack でロングに戻す
# (F) 結果が元データの集計に整合するか（例: groupby合計）を確認するコードを書いてください

stores = ["Tokyo","Osaka","Nagoya"]
cats   = ["A","B","C"]
years  = [2023, 2024, 2025]

data = []
for s in stores:
    for c in cats:
        for y in years:
            data.append([s, c, y, int(rng.integers(80,160))])
df_all = pd.DataFrame(data, columns=["store","category","year","amount"])

# ここに処理を書いてください。

#(A)
#print(df_all)

#(B)
df_all_mult = df_all.set_index(["store","category","year"])
#print(df_all_mult)

#(C)
df_all_pivot = df_all_mult.pivot_table(index=["store","category"],columns="year",values="amount",aggfunc="sum")
#print(df_all_pivot)

#(D)
df_all_pivot_year = df_all_pivot[2025]
#print(df_all_pivot_year)

#(E)
df_all_mult_un = df_all_mult.unstack(level="category")
print(df_all_mult_un)
df_all_mult_sta = df_all_mult_un.stack()
print(df_all_mult_sta)


            amount          
category         A    B    C
store  year                 
Nagoya 2023    104  153   93
       2024    153  153   92
       2025     91  127  102
Osaka  2023    150  127   95
       2024     85  108  145
       2025    118   91  105
Tokyo  2023     99  159  145
       2024    101  139  156
       2025     98  147   88


C:\Users\kawor\AppData\Local\Temp\ipykernel_18180\598796561.py:41: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  df_all_mult_sta = df_all_mult_un.stack()


In [ ]:

# === Q12: 追加チャレンジ（任意） ===
# Q11の df_all を使って、
# 1) 各 store × category の前年比成長率（2025 vs 2024 など）を計算してください。
# 2) 成長率トップ3の (store, category) を抽出してください。
# 3) 表示を見やすくするために、index/columns の names を設定・並べ替えを行ってください。

# ここに処理を書いてください。

